# 06 – Final model (all people)

Starts from `nhanes_model_ready.csv` (10 features from notebook 04) and adds four blood-pressure and cholesterol questions from `BPQ_L`. Then tunes a random forest and a gradient boosting model to see if they beat logistic regression.

**Note:** the final test is done in notebook 07, after removing people with a known diabetes diagnosis.

## 6.1 Load the 10-feature dataset

In [1]:
import pandas as pd
df=pd.read_csv('../data/processed/nhanes_model_ready.csv')
df

,age_years,is_female,bmi,waist_cm,hip_cm,waist_to_height,waist_to_hip,race_ethnicity_code,education_code,income_poverty_ratio,elevated_hbA1c
0,43,0,27.0,98.3,102.9,0.547632,0.955296,Asian,College graduate,5.00,0
1,66,0,33.5,114.7,112.4,0.658439,1.020463,White,College graduate,5.00,0
2,44,1,29.7,93.5,98.0,0.611511,0.954082,Other Hispanic,High school,1.41,1
3,34,0,30.2,106.1,110.6,0.612233,0.959313,Mexican American,Some college,1.33,0
4,68,1,42.6,122.0,148.9,0.782553,0.819342,White,College graduate,1.32,1
...,...,...,...,...,...,...,...,...,...,...,...
5762,69,1,27.9,111.0,104.5,0.697236,1.062201,Other or Multiracial,High school,0.98,1
5763,76,1,26.4,89.0,96.4,0.587847,0.923237,Mexican American,Less than 9th grade,2.25,1
5764,49,1,NaN,NaN,NaN,NaN,NaN,Black,College graduate,NaN,1
5765,40,0,25.5,96.0,103.3,0.536013,0.929332,Other Hispanic,Some college,3.11,0


In [2]:
df.shape

(5767, 11)

## 6.2 Load the blood pressure and cholesterol questions (`BPQ_L`)

In [3]:

import pandas as pd
from pathlib import Path

data_dir = Path("../data")

BPQ = pd.read_sas(data_dir / "BPQ_L.xpt", format="xport")

print(BPQ.shape)
BPQ.head()

(8501, 6)


,SEQN,BPQ020,BPQ030,BPQ150,BPQ080,BPQ101D
0,130378.0,1.0,1.0,1.0,2.0,2.0
1,130379.0,1.0,1.0,1.0,2.0,2.0
2,130380.0,2.0,NaN,NaN,1.0,1.0
3,130384.0,2.0,NaN,NaN,2.0,2.0
4,130385.0,2.0,NaN,NaN,2.0,2.0


## 6.3 Keep the columns we need

In [4]:
BPQ = BPQ[['SEQN', 'BPQ020', 'BPQ080', 'BPQ150', 'BPQ101D']]
print(BPQ.shape)

(8501, 5)


In [5]:
BPQ.isna().sum()

SEQN          0
BPQ020        3
BPQ080        3
BPQ150     5532
BPQ101D       3
dtype: int64

## 6.4 Attach the participant ID (`SEQN`): safety check first

**Why:** `nhanes_model_ready.csv` has no ID column. `nhanes_clean.csv` has it, in the same row order. If the BMI columns are identical, the IDs can be copied safely.

In [6]:
clean = pd.read_csv('../data/processed/nhanes_clean.csv')
print(df['bmi'].equals(clean['bmi']))

True


## 6.5 Merge the blood pressure and cholesterol questions

In [7]:
df['SEQN'] = clean['SEQN']
df = df.merge(BPQ, on='SEQN', how='left')
print(df.shape)

(5767, 16)


In [8]:
df.head()

,age_years,is_female,bmi,waist_cm,hip_cm,waist_to_height,waist_to_hip,race_ethnicity_code,education_code,income_poverty_ratio,elevated_hbA1c,SEQN,BPQ020,BPQ080,BPQ150,BPQ101D
0,43,0,27.0,98.3,102.9,0.547632,0.955296,Asian,College graduate,5.00,0,130378,1.0,2.0,1.0,2.0
1,66,0,33.5,114.7,112.4,0.658439,1.020463,White,College graduate,5.00,0,130379,1.0,2.0,1.0,2.0
2,44,1,29.7,93.5,98.0,0.611511,0.954082,Other Hispanic,High school,1.41,1,130380,2.0,1.0,NaN,1.0
3,34,0,30.2,106.1,110.6,0.612233,0.959313,Mexican American,Some college,1.33,0,130386,2.0,2.0,NaN,2.0
4,68,1,42.6,122.0,148.9,0.782553,0.819342,White,College graduate,1.32,1,130387,1.0,1.0,1.0,2.0


In [9]:
df['BPQ020'].value_counts()

BPQ020
2.0    3567
1.0    2196
9.0       4
Name: count, dtype: int64

## 6.5b Recode the answers to 0/1

1 = yes, 0 = no, other codes (don't know, refused) = missing. People without high blood pressure were not asked about medicine, so their `bp_med` is set to 0.

In [10]:
df['bp_told'] = df['BPQ020'].map({1: 1, 2: 0})
df['chol_told'] = df['BPQ080'].map({1: 1, 2: 0})
df['chol_med'] = df['BPQ101D'].map({1: 1, 2: 0})
df['bp_med'] = df['BPQ150'].map({1: 1, 2: 0})
df.loc[df['BPQ020'] == 2, 'bp_med'] = 0


In [11]:
new_cols = ['bp_told', 'chol_told', 'bp_med', 'chol_med']
print(df[new_cols].isna().sum())
print(df[new_cols].sum())

bp_told       4
chol_told    34
bp_med        7
chol_med     26
dtype: int64
bp_told      2196.0
chol_told    2400.0
bp_med       1827.0
chol_med     1659.0
dtype: float64


## 6.6 Drop helper columns, make X and y

In [12]:
df = df.drop(columns=['SEQN', 'BPQ020', 'BPQ080', 'BPQ150', 'BPQ101D'])
y = df.pop('elevated_hbA1c')
X = df
print(X.shape)

(5767, 14)


## 6.7 Split, preprocessor and Model C (cross-validation)

In [13]:
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.pipeline import make_pipeline
from sklearn.compose import make_column_transformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, recall_score, precision_score, roc_auc_score

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)

num_features = X_train.select_dtypes(include='number').columns.tolist()
print(num_features)

num_pipe = make_pipeline(SimpleImputer(strategy='median'), StandardScaler())
ord_pipe = make_pipeline(
    SimpleImputer(strategy='most_frequent'),
    OrdinalEncoder(categories=[['Less than 9th grade', '9th to 11th grade',
                                'High school', 'Some college', 'College graduate']]))
nom_pipe = make_pipeline(
    SimpleImputer(strategy='most_frequent'),
    OneHotEncoder(handle_unknown='ignore', sparse_output=False))

preprocessor = make_column_transformer(
    (num_pipe, num_features),
    (ord_pipe, ['education_code']),
    (nom_pipe, ['race_ethnicity_code']))

model_c = make_pipeline(preprocessor, LogisticRegression(max_iter=1000))

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_validate(model_c, X_train, y_train, cv=cv, scoring=['roc_auc', 'recall'])
print('ROC-AUC:', scores['test_roc_auc'].mean().round(3))
print('Recall :', scores['test_recall'].mean().round(3))

['age_years', 'is_female', 'bmi', 'waist_cm', 'hip_cm', 'waist_to_height', 'waist_to_hip', 'income_poverty_ratio', 'bp_told', 'chol_told', 'chol_med', 'bp_med']
ROC-AUC: 0.8
Recall : 0.607


## Model C: Model B plus 4 questions (5-fold cross-validation, training data)

| Model | Features | ROC-AUC | Recall |
|---|---|---|---|
| Model B | 10 | 0.787 | 0.587 |
| Model C | 14 (blood pressure and cholesterol questions added) | 0.800 | 0.607 |

## 6.8 Threshold: fewer missed cases (asymmetric classification)

In [14]:
from sklearn.model_selection import cross_val_predict

proba = cross_val_predict(model_c, X_train, y_train, cv=cv, method='predict_proba')[:, 1]

for t in [0.5, 0.4, 0.3, 0.25, 0.2]:
    pred = (proba >= t).astype(int)
    print('Threshold', t)
    print(confusion_matrix(y_train, pred))
    print('Recall:', round(recall_score(y_train, pred), 3),
          ' Precision:', round(precision_score(y_train, pred), 3))
    print()

Threshold 0.5
[[2239  535]
 [ 723 1116]]
Recall: 0.607  Precision: 0.676

Threshold 0.4
[[1958  816]
 [ 485 1354]]
Recall: 0.736  Precision: 0.624

Threshold 0.3
[[1652 1122]
 [ 293 1546]]
Recall: 0.841  Precision: 0.579

Threshold 0.25
[[1437 1337]
 [ 186 1653]]
Recall: 0.899  Precision: 0.553

Threshold 0.2
[[1214 1560]
 [ 110 1729]]
Recall: 0.94  Precision: 0.526



## 6.9 Threshold results (cross-validated, 4,613 training people: 1,839 elevated, 2,774 not)

| Threshold | Elevated people found (recall) | Healthy people wrongly flagged | Flagged who are truly elevated (precision) |
|---|---|---|---|
| 0.5 | 61% | 19% | 68% |
| 0.4 | 74% | 29% | 62% |
| 0.3 | 84% | 40% | 58% |
| 0.25 | 90% | 48% | 55% |
| 0.2 | 94% | 56% | 53% |

**Reading the table:** a lower threshold finds more elevated people, but also flags more healthy people.

## 6.10 RandomizedSearchCV: can a tuned random forest beat logistic regression?

**Note:** same 14 features, same five folds. Logistic regression to beat: ROC-AUC 0.800.

In [15]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier

# The council: first the preprocessor washes and cuts the data, then the forest votes.
rf_pipe = make_pipeline(preprocessor, RandomForestClassifier(random_state=42))

# The menu of knob positions. Name = step name + two underscores + knob name.
rf_params = {
    'randomforestclassifier__n_estimators': [200, 400],             # how many helpers
    'randomforestclassifier__max_depth': [4, 6, 8, 10, None],       # how many questions in a row
    'randomforestclassifier__min_samples_leaf': [5, 10, 20, 40],    # smallest group for a rule
    'randomforestclassifier__max_features': ['sqrt', 0.5],          # how many clues per question
}

# The competition: 20 random councils, 5 judges each, graded by ROC-AUC.
rf_search = RandomizedSearchCV(rf_pipe, rf_params, n_iter=20, cv=cv,
                               scoring='roc_auc', random_state=42, n_jobs=-1)

# Start the competition on the training people only.
rf_search.fit(X_train, y_train)

print('Best ROC-AUC:', round(rf_search.best_score_, 3))   # the winner's average grade
print('Best settings:', rf_search.best_params_)           # the winner's knob positions

Best ROC-AUC: 0.798
Best settings: {'randomforestclassifier__n_estimators': 400, 'randomforestclassifier__min_samples_leaf': 5, 'randomforestclassifier__max_features': 'sqrt', 'randomforestclassifier__max_depth': 8}


## 6.11 RandomizedSearchCV: can a tuned gradient boosting beat logistic regression?

**Note:** same 14 features, same five folds. Score to beat: 0.816 (0.800 plus the fold spread).

In [16]:
from sklearn.ensemble import HistGradientBoostingClassifier

# the relay team : the preprocessor first, then the boosting model.
gb_pipe= make_pipeline(preprocessor, HistGradientBoostingClassifier(random_state=42))

# The menu of knob positions (3 x 3 x 4 x 3 x 2 = 216 combinations).

gb_params = {
    'histgradientboostingclassifier__learning_rate': [0.02, 0.05, 0.1],   # size of each correction
    'histgradientboostingclassifier__max_iter': [100, 200, 400],          # how many helpers
    'histgradientboostingclassifier__max_depth': [2, 3, 4, None],         # questions per helper
    'histgradientboostingclassifier__min_samples_leaf': [20, 50, 100],    # smallest group for a rule
    'histgradientboostingclassifier__l2_regularization': [0.0, 1.0],      # the brake
}

# The competition: 20 random teams out of 216, 5 judges each.

gb_search= RandomizedSearchCV(gb_pipe, gb_params, n_iter=20, cv=cv,
                              scoring='roc_auc', random_state=43, n_jobs=-1)
gb_search.fit(X_train, y_train)

print('Best ROC_AUC:', round(gb_search.best_score_, 3))
print('Best settings:', gb_search.best_params_)

Best ROC_AUC: 0.798
Best settings: {'histgradientboostingclassifier__min_samples_leaf': 100, 'histgradientboostingclassifier__max_iter': 400, 'histgradientboostingclassifier__max_depth': 3, 'histgradientboostingclassifier__learning_rate': 0.02, 'histgradientboostingclassifier__l2_regularization': 0.0}


## 6.12 Tuned models vs logistic regression (14 features, same five folds)

| Model | Cross-validated ROC-AUC |
|---|---|
| Logistic regression (Model C) | **0.800** |
| Random forest, tuned (RandomizedSearchCV) | 0.798 |
| Gradient boosting, tuned (RandomizedSearchCV) | 0.798 |

**Decision:** all three perform the same (differences of 0.002 are far below the fold spread of about 0.016). We keep logistic regression because it is the simplest, gives a risk probability directly, and each factor can be explained as an odds ratio.

## 6.13 TPOT: does an automated search find a better pipeline?

**TPOT:** version 0.12.2 was installed, but the run failed on Python 3.14 with scikit-learn 1.9 (incompatible versions; TPOT crashed after the first generation).
Instead, RandomizedSearchCV tuned the random forest and gradient boosting (6.10, 6.11): both reached ROC-AUC 0.798, below logistic regression (0.800).

**Chosen model: logistic regression.** Its final version and the final test are in notebook 07.